# 1. Config e imports

In [1]:
!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.5/29.5 MB 51.2 MB/s eta 0:00:00


In [2]:
import os
import time
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator, AllChem
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator
from sklearn.metrics import f1_score
import gc
from torch.cuda.amp import autocast, GradScaler

# Configuración de dispositivo
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Utilizando dispositivo: {device}")

# Rutas de entrada (Asegúrate de haber añadido el dataset con los folds de la Fase 1)
TRAIN_FOLDS_PATH = "/kaggle/input/datasets/ernestomiguez/casmi2026-phase1-folds/train_folds.parquet"
TEST_PATH = "/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/test.parquet"
TRAIN_PATH = "/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/train.parquet"

Utilizando dispositivo: cuda


# 2. Dataset y Modelo PyTorch para Predicción de Fingerprints

In [3]:
class ResidualBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
            nn.ReLU(),
            nn.Dropout(0.3), # Dropout moderado para evitar sobreajuste rápido
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim)
        )
        self.relu = nn.ReLU()

    def forward(self, x):
        # Conexión residual: la entrada se suma a la salida del bloque
        return self.relu(x + self.block(x))

class TopKEncoderMLP(nn.Module):
    def __init__(self, top_k=64, input_dim=2, hidden_dim=512, output_dim=1024):
        super().__init__()
        self.top_k = top_k
        
        # Proyección inicial por cada pico (BatchNorm sobre hidden_dim, no sobre top_k)
        self.pico_mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim), 
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )
        
        # Capas globales para predecir el fingerprint de 1024 bits
        self.global_mlp = nn.Sequential(
            nn.Linear(hidden_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(1024, output_dim) # Devuelve logits
        )

    def forward(self, x):
            # x tiene forma: (Batch_Size, Top_K, 2) [m/z, intensity]
            batch_size = x.size(0)
            
            # Aplanar para aplicar la capa lineal a todos los picos a la vez: (Batch_Size * Top_K, input_dim)
            x_flat = x.view(-1, x.size(-1))
            out_picos = self.pico_mlp(x_flat)
            
            # Reestructurar de vuelta a (Batch_Size, Top_K, hidden_dim)
            out_picos = out_picos.view(batch_size, self.top_k, -1)
            
            # Agregación global (Max Pooling a lo largo de los picos para capturar los fragmentos clave)
            global_repr, _ = torch.max(out_picos, dim=1)
            
            # Predicción final de los 1024 bits
            logits = self.global_mlp(global_repr)
            return logits

# 3. Carga de Folds y Preparación de DataLoaders

In [4]:
# ==========================================
# CARGA Y PREPROCESAMIENTO SEGURO (5-FOLDS)
# ==========================================
print("Cargando dataset original y folds asignados...")
full_train_df = pd.read_parquet(TRAIN_PATH)
folds_df = pd.read_parquet(TRAIN_FOLDS_PATH)

if 'fold' in full_train_df.columns:
    full_train_df = full_train_df.drop(columns=['fold'])

# Unir para conservar espectros y el fold asignado de cada molécula
train_folds_df = full_train_df.merge(
    folds_df[['normalized_smiles', 'inchikey14', 'fold']].drop_duplicates(),
    on=['normalized_smiles', 'inchikey14'],
    how='inner'
)

# Liberar los dataframes intermedios individuales para ahorrar memoria RAM
del full_train_df, folds_df
gc.collect()

print(f"Dataset total combinado disponible para 5-Folds: {len(train_folds_df):,}")

# --- Definición de la Clase TopKLazyDataset (Carga Perezosa) ---
class TopKLazyDataset(Dataset):
    def __init__(self, df, top_k=64, n_bits=1024):
        self.df = df.reset_index(drop=True)
        self.top_k = top_k
        self.n_bits = n_bits
        self.morgan_gen = GetMorganGenerator(radius=2, fpSize=n_bits)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # 1. Procesar picos espectrales Top-K
        mz = np.array(row['ms2_mzs'], dtype=np.float32)
        intensity = np.array(row['ms2_normalized_intensities'], dtype=np.float32)
        
        if len(mz) > 0:
            sorted_idx = np.argsort(intensity)[::-1]
            mz = mz[sorted_idx][:self.top_k]
            intensity = intensity[sorted_idx][:self.top_k]
            
            max_int = intensity.max()
            if max_int > 0:
                intensity = intensity / max_int
        
        current_len = len(mz)
        if current_len < self.top_k:
            mz = np.pad(mz, (0, self.top_k - current_len), 'constant', constant_values=0)
            intensity = np.pad(intensity, (0, self.top_k - current_len), 'constant', constant_values=0)
            
        spectrum_features = np.stack([mz, intensity], axis=-1)
        x = torch.tensor(spectrum_features, dtype=torch.float32)
        
        # 2. Generar el fingerprint al vuelo de manera ultra rápida para este único SMILES
        smi = row['normalized_smiles']
        mol = Chem.MolFromSmiles(smi)
        if mol is not None:
            y_arr = self.morgan_gen.GetFingerprintAsNumPy(mol).astype(np.float32)
        else:
            y_arr = np.zeros((self.n_bits,), dtype=np.float32)
            
        y = torch.tensor(y_arr, dtype=torch.float32)
        
        return x, y

# --- Definición opcional de TopKTestDataset para la inferencia posterior ---
class TopKTestDataset(Dataset):
    def __init__(self, df, top_k=64):
        self.df = df.reset_index(drop=True)
        self.top_k = top_k

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        mz = np.array(row['ms2_mzs'], dtype=np.float32)
        intensity = np.array(row['ms2_normalized_intensities'], dtype=np.float32)
        
        if len(mz) > 0:
            sorted_idx = np.argsort(intensity)[::-1]
            mz = mz[sorted_idx][:self.top_k]
            intensity = intensity[sorted_idx][:self.top_k]
            max_int = intensity.max()
            if max_int > 0:
                intensity = intensity / max_int
                
        current_len = len(mz)
        if current_len < self.top_k:
            mz = np.pad(mz, (0, self.top_k - current_len), 'constant', constant_values=0)
            intensity = np.pad(intensity, (0, self.top_k - current_len), 'constant', constant_values=0)
            
        spectrum_features = np.stack([mz, intensity], axis=-1)
        return torch.tensor(spectrum_features, dtype=torch.float32)

# Cargar también el test dataframe y preparar su DataLoader de manera fija
test_df = pd.read_parquet(TEST_PATH)
test_dataset = TopKTestDataset(test_df, top_k=64)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=0, pin_memory=True)

print("✓ Dataset combinado y clases Dataset listas para el bucle de 5 Folds.")

Cargando dataset original y folds asignados...
Dataset total combinado disponible para 5-Folds: 2,539,608
✓ Dataset combinado y clases Dataset listas para el bucle de 5 Folds.


# 4. Instanciación del Modelo, Función de Pérdida y Optimizador

In [5]:
# Dimensión de entrada: num_bins (1000.0 / 0.5 = 2000) + 1 (precursor_mz)
OUTPUT_DIM = 1024  # Tamaño del Morgan Fingerprint


class CombinedLossWithLogits(nn.Module):
    def __init__(self, bce_weight=0.5, cosine_weight=0.5):
        super().__init__()
        # Usamos BCEWithLogitsLoss (seguro para autocast)
        self.bce_with_logits = nn.BCEWithLogitsLoss()
        self.bce_weight = bce_weight
        self.cosine_weight = cosine_weight
        
    def forward(self, logits, targets):
        # 1. Pérdida BCE directa sobre los logits
        loss_bce = self.bce_with_logits(logits, targets)
        
        # 2. Para el coseno, aplicamos sigmoide a los logits para obtener probabilidades entre 0 y 1
        preds = torch.sigmoid(logits)
        
        dot_product = (preds * targets).sum(dim=1)
        pred_norm = preds.norm(dim=1, p=2)
        target_norm = targets.norm(dim=1, p=2)
        
        cosine_sim = dot_product / (pred_norm * target_norm + 1e-8)
        loss_cosine = (1.0 - cosine_sim).mean()
        
        return (self.bce_weight * loss_bce) + (self.cosine_weight * loss_cosine)

# 5. Función de entrenamiento

In [6]:
def train_model_lightweight(model, train_loader, val_loader, optimizer, criterion, device, epochs=15, patience=3):
    best_val_loss = float('inf')
    best_model_wts = copy.deepcopy(model.state_dict())
    patience_counter = 0
    
    # Sintaxis actualizada para Mixed Precision
    scaler = torch.amp.GradScaler('cuda')

    print("Iniciando entrenamiento ligero antifugas de memoria...")
    
    for epoch in range(epochs):
        epoch_start_time = time.time()
        
        gc.collect()
        torch.cuda.empty_cache()
        
        # --- Fase de Entrenamiento ---
        model.train()
        train_loss = 0.0
        for x_batch, y_batch in train_loader:
            x_batch = x_batch.to(device, non_blocking=True)
            y_batch = y_batch.to(device, non_blocking=True)
            
            optimizer.zero_grad(set_to_none=True)
            
            with torch.amp.autocast('cuda'):
                logits = model(x_batch)
                loss = criterion(logits, y_batch)
                
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            train_loss += loss.item() * x_batch.size(0)
            
        epoch_train_loss = train_loss / len(train_loader.dataset)
        
        # --- Fase de Validación ---
        model.eval()
        val_loss = 0.0
        all_val_preds = []
        all_val_targets = []
        
        with torch.no_grad():
            for x_batch, y_batch in val_loader:
                x_batch = x_batch.to(device, non_blocking=True)
                y_batch = y_batch.to(device, non_blocking=True)
                
                with torch.amp.autocast('cuda'):
                    logits = model(x_batch)
                    loss = criterion(logits, y_batch)
                    
                val_loss += loss.item() * x_batch.size(0)
                
                preds_bin = (torch.sigmoid(logits).detach().cpu().numpy() > 0.5).astype(np.int8)
                targets_np = y_batch.detach().cpu().numpy().astype(np.int8)
                
                all_val_preds.append(preds_bin)
                all_val_targets.append(targets_np)
                
                del logits, preds_bin, targets_np
                
        epoch_val_loss = val_loss / len(val_loader.dataset)
        
        all_preds = np.concatenate(all_val_preds, axis=0)
        all_targets = np.concatenate(all_val_targets, axis=0)
        val_f1 = f1_score(all_targets, all_preds, average='micro', zero_division=0)
        
        del all_val_preds, all_val_targets, all_preds, all_targets
        torch.cuda.empty_cache()
        gc.collect()
        
        elapsed_time = time.time() - epoch_start_time
        print(f"Epoch [{epoch+1:02d}/{epochs:02d}] ({elapsed_time:.1f}s) | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | Val F1: {val_f1:.4f}")
        
        # --- Early Stopping ---
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            best_model_wts = copy.deepcopy(model.state_dict())
            patience_counter = 0
            print(f"  --> Guardado nuevo mejor modelo (Val Loss: {best_val_loss:.4f})")
        else:
            patience_counter += 1
            print(f"  --> Sin mejora en val loss ({patience_counter}/{patience})")
            if patience_counter >= patience:
                print(f"\nEarly Stopping activado en la época {epoch+1}.")
                break

    model.load_state_dict(best_model_wts)
    return model

# 6. Bucle de entrenamiento

In [7]:
# 1. Separar train y val usando únicamente el Fold 0
train_fold_df = train_folds_df[train_folds_df['fold'] != 0].reset_index(drop=True)
val_fold_df = train_folds_df[train_folds_df['fold'] == 0].reset_index(drop=True)

# 2. Instanciar Datasets y DataLoaders
train_dataset = TopKLazyDataset(train_fold_df, top_k=64)
val_dataset = TopKLazyDataset(val_fold_df, top_k=64)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

# Cargar también el test dataframe y preparar su DataLoader
test_df = pd.read_parquet(TEST_PATH)
test_dataset = TopKTestDataset(test_df, top_k=64)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False, num_workers=0, pin_memory=True)

# 3. Inicializar modelo, optimizador y pérdida
model = TopKEncoderMLP(top_k=64, input_dim=2, hidden_dim=512, output_dim=1024).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-3)
criterion = CombinedLossWithLogits(bce_weight=0.5, cosine_weight=0.5)

print("Iniciando entrenamiento rápido de Fold Único...")
# 4. Entrenar usando tu función optimizada (reducido a 10 épocas para gastar pocos recursos)
trained_model = train_model_lightweight(
    model=model, 
    train_loader=train_loader, 
    val_loader=val_loader, 
    optimizer=optimizer, 
    criterion=criterion, 
    device=device, 
    epochs=10, 
    patience=2
)

# 5. Guardar los pesos del modelo
torch.save(trained_model.state_dict(), 'fingerprint_mlp_fold0.pth')
print("✓ Pesos guardados en 'fingerprint_mlp_fold0.pth'")

# 6. Inferencia inmediata en Test y guardado del archivo final
trained_model.eval()
test_preds = []
with torch.no_grad():
    for x_batch in test_loader:
        x_batch = x_batch.to(device, non_blocking=True)
        with torch.amp.autocast('cuda' if 'cuda' in str(device) else 'cpu'):
            logits = trained_model(x_batch)
            preds = torch.sigmoid(logits)
        test_preds.append(preds.detach().cpu().numpy())

test_preds_arr = np.concatenate(test_preds, axis=0)
np.save('test_predicted_fingerprints.npy', test_preds_arr)

print("\n🎉 ¡Entrenamiento y predicción de Test completados con éxito!")
print(f"Forma de las predicciones: {test_preds_arr.shape}")

Iniciando entrenamiento rápido de Fold Único...
Iniciando entrenamiento ligero antifugas de memoria...
Epoch [01/10] (1014.4s) | Train Loss: 0.3275 | Val Loss: 0.3230 | Val F1: 0.3250
  --> Guardado nuevo mejor modelo (Val Loss: 0.3230)
Epoch [02/10] (1008.1s) | Train Loss: 0.3217 | Val Loss: 0.3203 | Val F1: 0.3307
  --> Guardado nuevo mejor modelo (Val Loss: 0.3203)
Epoch [03/10] (1016.8s) | Train Loss: 0.3198 | Val Loss: 0.3241 | Val F1: 0.3364
  --> Sin mejora en val loss (1/2)
Epoch [04/10] (1011.7s) | Train Loss: 0.3188 | Val Loss: 0.3181 | Val F1: 0.3370
  --> Guardado nuevo mejor modelo (Val Loss: 0.3181)
Epoch [05/10] (1000.9s) | Train Loss: 0.3180 | Val Loss: 0.3256 | Val F1: 0.3376
  --> Sin mejora en val loss (1/2)
Epoch [06/10] (1030.5s) | Train Loss: 0.3175 | Val Loss: 0.3179 | Val F1: 0.3330
  --> Guardado nuevo mejor modelo (Val Loss: 0.3179)
Epoch [07/10] (1022.2s) | Train Loss: 0.3171 | Val Loss: 0.3933 | Val F1: 0.3145
  --> Sin mejora en val loss (1/2)
Epoch [08/10]